# Best Practices for Creating Embeddings in HANA Cloud with SAP GenAI Hub (Orchestration SDK)

## Introduction
This notebook demonstrates creating text embeddings and storing them in SAP HANA Cloud using the **SAP Cloud SDK for AI Orchestration** module. It uses `OrchestrationConfig` with `Template` and `LLM` for answer generation, while leveraging the native embedding API for vector creation.

## Dataset
The sample corpus is a science and nutrition dataset (`science-data-sample.csv`). The similarity search query at the end is aligned with this corpus.

## Prerequisites
- SAP HANA Cloud instance with vector engine enabled
- SAP AI Core with a deployed embedding model (default: `text-embedding-3-small`)
- Orchestration service deployed on AI Core
- Environment variables configured in a `.env` file (see `.env-example`)

## Next Step
After running this notebook, use `Orchestration_RAG.ipynb` to query the stored embeddings with a RAG pipeline using OrchestrationConfig.

In [ ]:
# Import section
import os
import json
import math
import pandas as pd
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

from hana_ml import ConnectionContext
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from gen_ai_hub.proxy.native.openai import embeddings
from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from hdbcli import dbapi

In [ ]:
# Load CSV file
csv_path = './science-data-sample.csv'
df = pd.read_csv(csv_path, low_memory=False)
df.head()

In [ ]:
# Define columns
METADATA_COLS = ["Difficulty Level", "Category"]  # Metadata columns
TEXT_COL = "Topic"  # Document text
VECTOR_COL = "MY_VECTOR"  # Embedding column

In [ ]:
# Function to split text into smaller chunks
# Based on the document structure, the chunking strategy can be changed.
def chunk_text(text, chunk_size=500):
    """Splits text into fixed-length chunks."""
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

In [ ]:
# Process data for embeddings
processed_rows = []
for _, row in df.iterrows():
    metadata = {col: str(row[col]) for col in METADATA_COLS}  # Convert metadata to JSON
    chunks = chunk_text(str(row[TEXT_COL]))  # Chunk text
    for chunk in chunks:
        processed_rows.append([chunk, json.dumps(metadata)])  # Store text & metadata JSON

In [ ]:
# Create processed DataFrame
processed_df = pd.DataFrame(processed_rows, columns=["MY_TEXT", "MY_METADATA"])

processed_df.head()

In [ ]:
# Initialize GenAI Hub Proxy Client to access models
proxy_client = get_proxy_client('gen-ai-hub')

# Embedding model: configurable via environment variable
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")

# Function for batch-wise embedding generation for better performance.
def get_batch_embeddings(text_list, model=EMBEDDING_MODEL):
    """Generates embeddings in batch."""
    response = embeddings.create(model_name=model, input=text_list)
    return [res.embedding for res in response.data]

In [ ]:
# Process embeddings in batches
BATCH_SIZE = 100  # Set batch size
vectors = []

for i in range(0, len(processed_df), BATCH_SIZE):
    batch_texts = processed_df["MY_TEXT"].iloc[i:i+BATCH_SIZE].tolist()
    batch_embeddings = get_batch_embeddings(batch_texts)
    vectors.extend(batch_embeddings)

# Add embeddings to DataFrame
processed_df[VECTOR_COL] = vectors

In [ ]:
# Connect to SAP HANA
cc = ConnectionContext(
    address=os.environ.get("HANA_ADDRESS"),
    port=os.environ.get("HANA_PORT"),
    user=os.environ.get("HANA_USER"),
    password=os.environ.get("HANA_PASSWORD"),
    encrypt=True
)
print(cc.hana_version())
print(cc.get_current_schema())

cursor = cc.connection.cursor()

In [ ]:
# Create table in SAP HANA
TABLE_NAME = "SCIENCE_DATA"

# Create the table if it does not already exist
sql_command = f'''
CREATE TABLE {TABLE_NAME} (
    MY_TEXT NCLOB,
    MY_METADATA NCLOB,
    MY_VECTOR REAL_VECTOR
);
'''
try:
    cursor.execute(sql_command)
    print(f"Table {TABLE_NAME} created successfully.")
except Exception as e:
    print(f"Table {TABLE_NAME} already exists, proceeding with existing table.")

# WARNING: Uncomment the following lines to drop and recreate the table.
# This will delete all existing data.
# cursor.execute(f'DROP TABLE {TABLE_NAME}')
# cursor.execute(sql_command)

cursor.close()

In [ ]:
# Prepare data for insertion
processed_df["MY_VECTOR"] = processed_df["MY_VECTOR"].apply(json.dumps)  # Change to acceptable format to consider as REAL_VECTOR
data = processed_df.values.tolist()

# Define batch size
BATCH_SIZE = 100

# Insert data into SAP HANA
cursor = cc.connection.cursor()
sql_insert = f'''
    INSERT INTO {TABLE_NAME}
    (MY_TEXT, MY_METADATA, MY_VECTOR)
    VALUES (?, ?, TO_REAL_VECTOR(?))
'''

# Process insertion in batches for better performance
total_batches = math.ceil(len(data) / BATCH_SIZE)

for i in range(total_batches):
    batch_data = data[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
    try:
        cursor.executemany(sql_insert, batch_data)
        cc.connection.commit()
        print(f"Inserted batch {i + 1}/{total_batches}")
    except Exception as e:
        print(f"Error inserting batch {i + 1}: {e}")
cursor.close()

## Verify Embeddings with Orchestration SDK

Use the Orchestration SDK with `OrchestrationConfig` to perform a vector search and generate a grounded answer using the stored embeddings.

In [ ]:
# Function to get embeddings for a query
def get_embedding(query):
    """Get embedding vector for a given text."""
    embeds = embeddings.create(
        model_name=EMBEDDING_MODEL,
        input=query
    )
    return embeds.data[0].embedding

In [ ]:
# Function to perform vector search
def run_vector_search(query, cursor, table_name, metric="COSINE_SIMILARITY", k=4):
    """Performs vector search on indexed documents."""
    ALLOWED_METRICS = {"COSINE_SIMILARITY", "L2DISTANCE"}
    if metric not in ALLOWED_METRICS:
        raise ValueError(f"metric must be one of {ALLOWED_METRICS}")
    k = int(k)

    query_vector = get_embedding(query)
    if not query_vector:
        raise ValueError("Failed to generate query embedding.")

    sort_order = "DESC" if metric != "L2DISTANCE" else "ASC"
    sql_query = f'''
    SELECT TOP {k} MY_TEXT, MY_METADATA
    FROM "{table_name}"
    ORDER BY {metric}(MY_VECTOR, TO_REAL_VECTOR('{query_vector}')) {sort_order}
    '''
    cursor.execute(sql_query)
    return cursor.fetchall()

In [ ]:
# Initialize Orchestration Service
orchestration_service = OrchestrationService()

# Configure LLM
llm = LLM(name="gpt-4o", parameters={"temperature": 0.0})

# Define prompt template with placeholders for context and query
prompt_template = Template(messages=[
    SystemMessage("You are a helpful assistant. Use the given context to answer the user's query accurately."),
    UserMessage("""Context: {{?context}}

Based on the above context, answer the following query:
{{?query}}

The answer tone has to be very professional in nature.
If you don't know the answer, politely say that you don't know, don't try to make up an answer."""),
])

# Build Orchestration Config
config = OrchestrationConfig(template=prompt_template, llm=llm)

In [ ]:
# Execute vector search and generate answer using Orchestration SDK
query = "How can you test for the presence of proteins in food?"

cursor = cc.connection.cursor()
context_records = run_vector_search(query, cursor, TABLE_NAME, "COSINE_SIMILARITY", 4)
context = " ".join([c[0] for c in context_records])

# Run orchestration with retrieved context
response = orchestration_service.run(
    config=config,
    template_values=[
        TemplateValue(name="context", value=context),
        TemplateValue(name="query", value=query),
    ]
)

print(response.orchestration_result.choices[0].message.content)